In [ ]:
import os
import torch
from dotenv import load_dotenv
from openai import OpenAI
from transformers import AutoTokenizer, AutoModelForCausalLM



In [ ]:
# openrouter setup
load_dotenv(override=True)

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY is not set in the environment variables.")

base_url = "https://openrouter.ai/api/v1"
openrouter = OpenAI(api_key=OPENROUTER_API_KEY, base_url=base_url)


In [ ]:
# huggingface setup
hf_token = os.getenv("HF_TOKEN")
if not hf_token:
    raise ValueError("HF_TOKEN is not set in the environment variables.")

LLAMA_MODEL = "meta-llama/Llama-3.2-3B-Instruct"

In [ ]:
# building prompt for synthetic data generation

system_prompt = f'''
You are a Synthetic Data Generation Assistant.
Your responsibility is to generate realistic, diverse and completely artificial datasets based on the user's requirements.

Follow these instructions:

Understand the user's request, including the type of data, number of records, required fields and any constraints.

Generate the exact number of records requested.

Ensure all generated records follow a consistent structure and data types.

Generate realistic and logically consistent values.

Maintain diversity across records and avoid unnecessary duplicates.

Respect all user-defined constraints, ranges and relationships between fields.

Never use actual personal information or claim that generated records belong to real individuals.

Return the generated dataset as valid JSON.

Do not include Markdown code fences, explanations, introductions or additional text outside the JSON.

If the request is ambiguous or missing essential information, make reasonable assumptions where possible.
'''



In [ ]:
def build_prompt(industry,sample_size,description):

    user_prompt = f'''
    Generate a synthetic dataset for the {industry} industry.
    The dataset should contain {sample_size} records.
    Description: {description}
  
    '''
    message = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    return message

In [ ]:
def generate_with_openrouter(message,model):
    response = openrouter.chat.completions.create(
        model=model,
        messages=message,
        temperature=0.7,
        max_tokens=1000
    )
    return response.choices[0].message.content

In [ ]:
def generate_with_huggingface(message,model):

    device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == "cuda":
        print("CUDA is available. Using GPU for inference.")
    else:
        print("CUDA is not available. Using CPU for inference.")

    tokenizer = AutoTokenizer.from_pretrained(model)
    tokenizer.pad_token = tokenizer.eos_token
    inputs= tokenizer.apply_chat_template(
        message,
        return_tensors="pt",
        add_generation_prompt=True,
        return_dict=True
        )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    model_instance = AutoModelForCausalLM.from_pretrained(model,dtype=torch.float16).to(device)
    outputs = model_instance.generate(**inputs, max_new_tokens=1200, temperature=0.7)

    generated_text = tokenizer.decode(
            outputs[0][inputs['input_ids'].shape[1]:],
            skip_special_tokens=True
        )
    return generated_text
    

In [ ]:
import json
import re

def validate_json(generated_data):
    try:
        generated_data = generated_data.strip()

        if not generated_data:
            return None, "Model returned an empty response"

        # Remove Markdown code fences
        generated_data = re.sub(
            r"^```(?:json)?\s*",
            "",
            generated_data,
            flags=re.IGNORECASE
        )

        generated_data = re.sub(
            r"\s*```$",
            "",
            generated_data
        )

        # Parse JSON
        json_object = json.loads(generated_data)

        return json_object, None

    except json.JSONDecodeError as e:
        return None, f"Invalid JSON: {str(e)}"


In [ ]:
def generate_synthetic_data(industry, sample_size, description, model_source, model_name):
    build_prompt(industry, sample_size, description)
    message = build_prompt(industry, sample_size, description)
    try:
         if model_source.lower() == "openrouter":
                generated_data = generate_with_openrouter(message, model_name)
         else:
                generated_data = generate_with_huggingface(message, model_name)
         print("Generated data type:", type(generated_data))
         print("Generated data length:", len(generated_data))
         print("Generated data:", repr(generated_data))
         data, error = validate_json(generated_data)
         if error:
            return  f"Error validating JSON:\n {error}"
         return f"Generated successfully {json.dumps(data,indent=4)}"
    except Exception as e:
        return f"Error generating data: \n{str(e)}"
   


In [ ]:
import gradio as gr

hf_models = ["meta-llama/Llama-3.2-1B-Instruct", "google/gemma-3-270m-it","Qwen/Qwen3-4B-Instruct-2507","deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B", "microsoft/Phi-3-mini-4k-instruct"]
openrouter_models = ["openai/gpt-4o-mini"]

with gr.Blocks() as demo:
    gr.Markdown("# Synthetic Data Generator")
    industry = gr.Textbox(label="Industry")
    sample_size = gr.Textbox(label="Sample Size")
    description = gr.Textbox(label="Description")

    
    model_source = gr.Radio(
        choices=["OpenRouter", "HuggingFace"],
        label="Select Model Source",
        value="HuggingFace"
    )

    model_name = gr.Dropdown(
            choices=hf_models,
            value=hf_models[0],
            label="Select Model",
            visible=True
        )

    def update_model_choices(model_source):
        if model_source == "OpenRouter":
            return gr.update(choices=openrouter_models, value=openrouter_models[0],visible=True)
        else:
            return gr.update(choices=hf_models, value=hf_models[0],visible=True)
    model_source.change(fn=update_model_choices, inputs=model_source, outputs=model_name)

    generate_button = gr.Button("Generate")

    output = gr.Code(label="Generated Synthetic Data",
    language="json",
    lines=25,
    interactive=False)
    
  
    generate_button.click(fn=generate_synthetic_data, inputs=[industry,sample_size,description,model_source,model_name], outputs=[output])
demo.launch()